<a href="https://colab.research.google.com/github/Text-Machine/temporal-adapters/blob/train/temporal-sft-train.ipynb" target="_parent\"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

# Notebook for data preperation

In [2]:
from datasets import Dataset, load_dataset

In [3]:
dataset = load_dataset("Kaspar/key_phrases_dataset")

README.md:   0%|          | 0.00/453 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.04M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
# Inspect the chat structure of prompt and completion
ex = dataset['train'][0]
print("Prompt messages:")
for msg in ex['prompt']:
    print(f"  role={msg.get('role','?')}, content={msg['content'][:120]}...")
print("\nCompletion messages:")
for msg in ex['completion']:
    print(f"  role={msg.get('role','?')}, content={msg['content'][:120]}...")

Prompt messages:
  role=user, content=Generate an article published in 1890 given the key phrases ['John Savage funeral', 'typhoid fever death', 'cemetery bur...

Completion messages:
  role=assistant, content=THE LATE MR. JOHN SAVAGE.

The funeral of the late Mr. John Savage took
place at the cemetery on Saturday afternoon. The...


In [7]:
print(dataset)
print("---")
print("Columns:", dataset['train'].column_names)
print("Num rows (train):", len(dataset['train']))
print("---")
print("First example:")
for k, v in dataset['train'][0].items():
    print(f"  {k}: {repr(v)[:200]}")

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1000
    })
})
---
Columns: ['prompt', 'completion']
Num rows (train): 1000
---
First example:
  prompt: [{'content': 'Generate an article published in 1890 given the key phrases [\'John Savage funeral\', \'typhoid fever death\', \'cemetery burial\', \'Rev. Thomas Rigby\', "St. Peter\'s Church sidesman",
  completion: [{'content': "THE LATE MR. JOHN SAVAGE.\n\nThe funeral of the late Mr. John Savage took\nplace at the cemetery on Saturday afternoon. The\ndeceased, who was the third son of the late Mr. Wm.\nSavage, 


## Setup: Install dependencies

Install the required libraries for SFT with LoRA adapters on Gemma 3 1B IT.

In [ ]:
!pip install -q trl peft transformers accelerate bitsandbytes datasets

## Prepare the dataset for SFT

The dataset has `prompt` (user messages) and `completion` (assistant messages) columns in chat format. TRL's `SFTTrainer` expects a single `messages` column containing the full conversation. We merge prompt + completion into one list of messages per example.

In [ ]:
def merge_prompt_completion(example):
    """Merge prompt and completion messages into a single messages list."""
    messages = example["prompt"] + example["completion"]
    return {"messages": messages}

sft_dataset = dataset["train"].map(merge_prompt_completion, remove_columns=["prompt", "completion"])

# Verify
print("Columns:", sft_dataset.column_names)
print("Example messages:")
for msg in sft_dataset[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

## Load Model and Tokenizer

Load the Gemma 3 1B IT model with 4-bit quantization (QLoRA) to reduce memory usage. We configure a LoRA adapter targeting the attention projection layers.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

model_id = "google/gemma-3-1b-it"

# 4-bit quantization config for QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",
)

print(f"Model loaded: {model_id}")
print(f"Model dtype: {model.dtype}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

## Configure LoRA

Set up the LoRA adapter configuration. We target the query and value projection layers in the attention blocks, matching the pattern used in the existing adapters in this repo.

In [ ]:
# LoRA configuration matching existing repo adapters
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

print("LoRA config:")
print(f"  rank (r): {lora_config.r}")
print(f"  alpha: {lora_config.lora_alpha}")
print(f"  target modules: {lora_config.target_modules}")
print(f"  dropout: {lora_config.lora_dropout}")

## Configure SFT Training

Set up training arguments and the `SFTTrainer` from TRL. Key choices:
- **Chat template**: handled automatically by `SFTTrainer` when a `messages` column is present
- **Completions-only training**: we only compute loss on the assistant's response, not the user prompt
- **Gradient checkpointing** and **bf16** for memory efficiency

In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

# Response template — this marks the beginning of the assistant's turn in Gemma's chat format.
# SFTTrainer will only compute loss on tokens after this template.
response_template = "<start_of_turn>model\n"

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

output_dir = "./sft-gemma-3-1b-keyphrase"

sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    max_seq_length=1024,
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="none",
)

print(f"Output dir: {output_dir}")
print(f"Epochs: {sft_config.num_train_epochs}")
print(f"Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"Max seq length: {sft_config.max_seq_length}")

## Create Trainer and Run SFT

Instantiate the `SFTTrainer` with the model, LoRA config, dataset, and completions-only data collator. Then launch training.

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
    data_collator=collator,
)

print(f"Trainable parameters: {trainer.model.print_trainable_parameters()}")

In [ ]:
# Launch training
train_result = trainer.train()

# Print final metrics
print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

## Save the fine-tuned adapter

Save the trained LoRA adapter weights and tokenizer locally. Optionally push to the Hugging Face Hub.

In [ ]:
# Save adapter locally
final_adapter_dir = f"{output_dir}/final-adapter"
trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"Adapter saved to: {final_adapter_dir}")

# Optional: push to Hugging Face Hub
# trainer.model.push_to_hub("your-username/sft-gemma-3-1b-keyphrase")
# tokenizer.push_to_hub("your-username/sft-gemma-3-1b-keyphrase")